# 🚀 Phase 2A: Track A Few-Shot & Zero-Day Generalization Benchmark ($N \le 10\text{k}$)
## *Task-Technology Fit Analysis of Modern AI-Driven Intrusion Detection: An Axiomatic-Empirical Fuzzy DEMATEL Simulation Framework*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

---

### 📌 Track A Benchmark Objectives:
1. **Multi-Paradigm Comparative Evaluation**: Benchmark 8 modern architectures across 5-fold cross-validation:
   - **Foundation Models**: `TabPFN v3`, `TabICL v2` (Bayesian In-Context Zero-Shot Learning)
   - **Modern Deep Learning**: `Mambular SSM` ($O(L)$ linear scaling), `FT-Transformer` ($O(L^2)$ attention), `SAINT` (dual self/row attention), `GraphIDS` (Inductive GNN)
   - **Tuned Baselines**: `XGBoost`, `LightGBM` (Optuna 50 trials)
2. **Anti-Leakage Protection**: Strict `GroupKFold` partitioning by `/24` subnet masks with fold-isolated preprocessing.
3. **Autorecovery Checkpointing**: Powered by `CheckpointManager` on Google Drive—survives sudden disconnects and preemptions without restarting from fold 1.
4. **Task-Technology Fit (TTF) Optimization**: Calculates empirical utility scores for $T_1$ (Line-Rate Edge), $T_2$ (Zero-Day Payload Isolation), and $T_3$ (Multi-Host Tracking).
5. **Publication-Ready Exports**: Generates LaTeX performance tables and 300+ DPI vector figures.


### 1. ☁️ Google Drive Mount & Workspace Initialization


In [ ]:
import os, sys
from pathlib import Path

# 1. Mount Google Drive for persistent state & checkpoint recovery
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/is_ai-vuln')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    
    # Navigate to repository directory
    if Path('/content/is_ai-vuln').exists():
        os.chdir('/content/is_ai-vuln')
    elif DRIVE_ROOT.exists() and (DRIVE_ROOT / 'src').exists():
        os.chdir(str(DRIVE_ROOT))
    print(f"✅ Google Drive mounted. Active directory: {os.getcwd()}")
except ImportError:
    print("ℹ️ Running in local/workstation environment.")

# Ensure workspace root is in sys.path
if '.' not in sys.path:
    sys.path.insert(0, '.')
print(f"Python: {sys.version.split()[0]}")


### 2. 📦 Core & Model Dependencies Installation


In [ ]:
# Install core dependencies for Track A benchmark
!pip install -q xgboost lightgbm optuna scikit-learn imbalanced-learn pandas numpy matplotlib seaborn networkx requests tqdm

# Install tabular foundation & deep tabular packages (if running on Colab)
# !pip install -q tabpfn mambular

print("✅ Benchmark dependencies ready.")


### 3. 🛡️ Data Ingestion, Decontamination & Anti-Leakage Partitioning


In [ ]:
import pandas as pd
from src.data.prep_pipeline import run_preparation_pipeline
from src.data.drive_downloader import initialize_dataset_directories

# Ensure persistent directories exist
dirs = initialize_dataset_directories()

# Run or load preprocessed decontaminated benchmark dataset
prep_result = run_preparation_pipeline("CICIDS2017", prefer_sample=True, n_splits=5)
print("Data Preparation Status:", prep_result["clean_stats"])


### 4. 🔄 Fault-Tolerant Checkpoint Bootstrap (`CheckpointManager`)


In [ ]:
import json
from src.utils.checkpoint_manager import CheckpointManager

# Establish checkpoint directory in Google Drive or workspace cache
chk_dir = Path("/content/drive/MyDrive/is_ai-vuln/checkpoints") if Path("/content/drive/MyDrive").exists() else Path("./workspace_drive/checkpoints")
chk_dir.mkdir(parents=True, exist_ok=True)

manager = CheckpointManager(
    drive_checkpoint_dir=chk_dir,
    dataset_name="CICIDS2017",
    track_name="Track_A",
    total_folds=5
)

print(f"🔄 Checkpoint State: {manager.state['status']}")
print(f"Completed models: {manager.state['completed_models']}")
print(f"Current model: {manager.state['current_model']} (Fold: {manager.state['current_fold']})")


### 5. 🔬 Track A 5-Fold Cross-Validation Execution (8 Models)


In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from src.models import get_model
from src.evaluation import evaluate_fold_run, calculate_ttf_utility
from src.data.splitters import AntiLeakageGroupKFold, extract_subnet_mask, safe_slice
from src.utils.environment import flush_memory

# Load cleaned dataset
clean_file = prep_result["processed_file"]
df = pd.read_parquet(clean_file) if str(clean_file).endswith(".parquet") else pd.read_csv(clean_file)

# Prepare features and labels (Sample N=10,000 for Track A)
df_track_a = df.head(10000).copy()
target_col = "is_attack" if "is_attack" in df_track_a.columns else "label"
feature_cols = [c for c in df_track_a.select_dtypes(include=[np.number]).columns if c != target_col]

X = df_track_a[feature_cols].values
y = df_track_a[target_col].values

# Extract subnet blocks for host session leakage protection
src_ip_col = "source_ip" if "source_ip" in df_track_a.columns else None
subnets = extract_subnet_mask(df_track_a[src_ip_col]) if src_ip_col else pd.Series(np.arange(len(df_track_a)) // 2000)

models_to_run = [
    "XGBoost",
    "LightGBM",
    "TabPFN_v3",
    "TabICL_v2",
    "Mambular_SSM",
    "FT_Transformer",
    "SAINT",
    "GraphIDS"
]

gkf = AntiLeakageGroupKFold(n_splits=5)
fold_indices = list(gkf.split(X, y, groups=subnets))

all_metrics = []

for model_name in models_to_run:
    if manager.should_skip_model(model_name):
        print(f"⏩ Model '{model_name}' already fully completed in checkpoint. Skipping...")
        continue
        
    print(f"\n{'='*50}\n🚀 Running Model: {model_name}\n{'='*50}")
    
    for fold_idx, (train_idx, val_idx) in enumerate(fold_indices, start=1):
        if manager.should_skip_fold(model_name, fold_idx):
            print(f"  ⏩ Fold {fold_idx} already cached. Skipping...")
            continue
            
        print(f"  ▶️ Training {model_name} | Fold {fold_idx}/5 (Train: {len(train_idx)}, Val: {len(val_idx)})...")
        
        # 1. Fold-isolated scaling & balancing (Anti-Leakage)
        X_tr, y_tr = safe_slice(X, train_idx), safe_slice(y, train_idx)
        X_va, y_va = safe_slice(X, val_idx), safe_slice(y, val_idx)
        
        scaler = StandardScaler().fit(X_tr)
        X_tr_sc = scaler.transform(X_tr)
        X_va_sc = scaler.transform(X_va)
        
        # 2. Fit Model
        model = get_model(model_name)
        model.fit(X_tr_sc, y_tr)
        
        # 3. Predict & Profile
        preds = model.predict(X_va_sc)
        probs = model.predict_proba(X_va_sc)
        profile = model.profile_inference(X_va_sc, warmup_runs=3, repeat_runs=10)
        
        # 4. Compute Metrics & TTF Utility
        fold_met = evaluate_fold_run(y_va, preds, probs, profile)
        fold_met["ttf_t1"] = calculate_ttf_utility(fold_met["f1_macro"], fold_met["latency_ms_per_flow"], 0.05, fold_met["vram_peak_mb"], 10.0, task="T1")
        fold_met["ttf_t2"] = calculate_ttf_utility(fold_met["f1_macro"], fold_met["latency_ms_per_flow"], 0.05, fold_met["vram_peak_mb"], 10.0, task="T2")
        fold_met["ttf_t3"] = calculate_ttf_utility(fold_met["f1_macro"], fold_met["latency_ms_per_flow"], 0.05, fold_met["vram_peak_mb"], 10.0, task="T3")
        
        print(f"    ✅ Fold {fold_idx} Result: Macro F1={fold_met['f1_macro']:.4f}, Latency={fold_met['latency_ms_per_flow']}ms, TTF(T1)={fold_met['ttf_t1']:.4f}")
        
        # 5. Checkpoint & Memory Cleanup
        manager.record_fold_completion(model_name, fold_idx, fold_met)
        model.cleanup()
        flush_memory()

manager.mark_completed()
print("\n🏆 Track A Benchmark Successfully Completed!")


### 6. 📊 Performance Aggregation & LaTeX Table Export


In [ ]:
# Extract full metrics from CheckpointManager
results_list = []
for model_name, folds_dict in manager.state.get("metrics_accumulator", {}).items():
    f1_list = [v["f1_macro"] for v in folds_dict.values()]
    lat_list = [v["latency_ms_per_flow"] for v in folds_dict.values()]
    vram_list = [v["vram_peak_mb"] for v in folds_dict.values()]
    ttf1_list = [v.get("ttf_t1", 0.0) for v in folds_dict.values()]
    ttf2_list = [v.get("ttf_t2", 0.0) for v in folds_dict.values()]
    
    results_list.append({
        "Model Architecture": model_name,
        "F1-Macro (Mean ± Std)": f"{np.mean(f1_list):.4f} ± {np.std(f1_list):.4f}",
        "Latency ms/flow": f"{np.mean(lat_list):.4f}",
        "Peak VRAM (MB)": f"{np.mean(vram_list):.1f}",
        "TTF Utility (T1: Edge)": f"{np.mean(ttf1_list):.4f}",
        "TTF Utility (T2: Zero-Day)": f"{np.mean(ttf2_list):.4f}"
    })

results_df = pd.DataFrame(results_list)
display(results_df)

# Export LaTeX table
out_dir = Path("./experiment_output/track_a_benchmark")
out_dir.mkdir(parents=True, exist_ok=True)
results_df.to_csv(out_dir / "performance_table.csv", index=False)
results_df.to_latex(out_dir / "performance_table.tex", index=False)
print(f"📄 Tables exported to: {out_dir.resolve()}")


### 7. 📈 Journal-Grade Visualization (300+ DPI Vector PDF)


In [ ]:
import matplotlib.pyplot as plt
from src.visualization.publication_styler import set_publication_style, save_publication_figure

set_publication_style(is_double_column=False)

fig, ax = plt.subplots()
models = list(manager.state.get("metrics_accumulator", {}).keys())
f1_means = [np.mean([v["f1_macro"] for v in manager.state["metrics_accumulator"][m].values()]) for m in models]
lat_means = [np.mean([v["latency_ms_per_flow"] for v in manager.state["metrics_accumulator"][m].values()]) for m in models]

scatter = ax.scatter(lat_means, f1_means, c='navy', s=80, edgecolors='black', alpha=0.85)
for i, txt in enumerate(models):
    ax.annotate(txt, (lat_means[i], f1_means[i]), textcoords="offset points", xytext=(5, 5), fontsize=8)

ax.set_title("Track A: Line-Rate Latency vs Macro F1 Pareto Frontier")
ax.set_xlabel("Inference Latency (ms / flow) [Lower is Better]")
ax.set_ylabel("Detection F1-Macro [Higher is Better]")
ax.grid(True, linestyle="--", alpha=0.4)

save_publication_figure(fig, str(out_dir / "figure_track_a_pareto_frontier"))
plt.show()
